In [10]:
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
import re

In [ ]:
pd.set_option("display.max_colwidth", None) ## 출력 잘림 해결

In [2]:
df = pd.read_excel("../data/myfair_participation_message_1982.xlsx").dropna(how='all')
df.head(3)

,id,message,process_state,sender_email,message_type,participation_num
0,43031.0,"<div class=""css-0""><br data-mce-bogus=""1""></di...",부스 예약 접수 완료,customer@customer.com,ADMIN,1982
1,43034.0,"<p>(고객사 담당자명 직함)님, 안녕하세요</p><p>마이페어 (마이페어 담당자명...",참가 품목 안내,myfair@myfair.co,ADMIN,1982
2,43435.0,"<p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus=""1...",부스 포함사항 안내,myfair@myfair.co,ADMIN,1982


# 전처리

In [5]:
def preprocess_html_message(html_text):
    if pd.isna(html_text):
        return ""  # 또는 NaN 유지하려면: return np.nan

    # BeautifulSoup으로 HTML 파싱
    soup = BeautifulSoup(html_text, "html.parser")

    # <br> 태그는 줄바꿈으로 대체
    for br in soup.find_all("br"):
        br.replace_with("\n")

    # 텍스트만 추출
    raw_text = soup.get_text(separator=" ", strip=True)

    # 공백 정리
    raw_text = re.sub(r'\s+', ' ', raw_text)

    return raw_text

In [6]:
df['message_clean'] = df['message'].apply(preprocess_html_message)

# 결과 출력
print(df[['message', 'message_clean']].head())

                                             message  \
0  <div class="css-0"><br data-mce-bogus="1"></di...   
1  <p>(고객사 담당자명 직함)님, 안녕하세요</p><p>마이페어 (마이페어 담당자명...   
2  <p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus="1...   
3  <p>담당자님 안녕하세요 보내주신 내용 확인했습니다<br>부재중이셔서 채팅드립니다<...   
4  <p>네 담당자님 확인했습니다!&nbsp;혹시 부스 자리 선택 또는 2면 오픈 부스...   

                                       message_clean  
0  🎉 부스예약이 접수되었습니다. 마이페어에서 참가신청을 시작해 주셔서 감사합니다. 서...  
1  (고객사 담당자명 직함)님, 안녕하세요 마이페어 (마이페어 담당자명)입니다. 비품이...  
2  안녕하세요, 마이페어입니다. 답변 기다려 주셔서 감사합니다. 부스패키지는 아래와 같...  
3  담당자님 안녕하세요 보내주신 내용 확인했습니다 부재중이셔서 채팅드립니다 부스 안에 ...  
4  네 담당자님 확인했습니다! 혹시 부스 자리 선택 또는 2면 오픈 부스 선택 가능한가...  


In [8]:
df[df['message_clean'].str.contains(r'[()]', na=False)]

,id,message,process_state,sender_email,message_type,participation_num,message_clean
0,43031.0,"<div class=""css-0""><br data-mce-bogus=""1""></di...",부스 예약 접수 완료,customer@customer.com,ADMIN,1982,🎉 부스예약이 접수되었습니다. 마이페어에서 참가신청을 시작해 주셔서 감사합니다. 서...
1,43034.0,"<p>(고객사 담당자명 직함)님, 안녕하세요</p><p>마이페어 (마이페어 담당자명...",참가 품목 안내,myfair@myfair.co,ADMIN,1982,"(고객사 담당자명 직함)님, 안녕하세요 마이페어 (마이페어 담당자명)입니다. 비품이..."
2,43435.0,"<p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus=""1...",부스 포함사항 안내,myfair@myfair.co,ADMIN,1982,"안녕하세요, 마이페어입니다. 답변 기다려 주셔서 감사합니다. 부스패키지는 아래와 같..."
11,43539.0,"<p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus=""1...",문의 답변,myfair@myfair.co,ADMIN,1982,"안녕하세요, 마이페어입니다. 바우처 75만원 (부가세 별도)에 대한 인보이스는 여기..."
13,43557.0,"<p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus=""1...",문의 답변,myfair@myfair.co,ADMIN,1982,"안녕하세요, 마이페어입니다. 문의주신 부분은 주최측에게 참가신청서 수령 후 확인 도..."
...,...,...,...,...,...,...,...
157,63094.0,"<p style="""">안녕하세요, (고객사 담당자명 직함)님.</p><p style...",참가 안내 (1),myfair@myfair.co,ADMIN,1982,"안녕하세요, (고객사 담당자명 직함)님. 마이페어 (마이페어 담당자명 호칭)입니다...."
159,63110.0,"<p style="""">안녕하세요, (고객사 담당자명 직함)님.</p><p style...",정산 안내,myfair@myfair.co,ADMIN,1982,"안녕하세요, (고객사 담당자명 직함)님. 마이페어 (마이페어 담당자명 호칭)입니다...."
161,63121.0,"<p style="""">안녕하세요, (고객사 담당자명 직함)님.</p><p style...",정산 안내 (1),myfair@myfair.co,ADMIN,1982,"안녕하세요, (고객사 담당자명 직함)님. 마이페어 (마이페어 담당자명 호칭)입니다...."
162,66016.0,"<p style="""">안녕하세요, (고객사 담당자명 직함)님.</p><p style...",수출바우처 정산 완료 안내,myfair@myfair.co,ADMIN,1982,"안녕하세요, (고객사 담당자명 직함)님. 마이페어 (마이페어 담당자명 호칭)입니다...."


In [14]:
df = df.drop(index=range(165, 171))  # 비어잇는 행 제거

***

# 비식별화

1. 마이페어 상담원 비식별화
- 현재 조건: "마이페어 홍길동입니다" → "마이페어 (마이페어 담당자명)입니다"

> 마이페어 담당자들은 대부분 "마이페어 + 이름 + 입니다" 패턴을 사용함

- 추가로 고려하면 좋은 점: "마이페어 홍길동 드림", "마이페어 김지훈 올림" 같은 변형 및 +"마이페어 홍길동입니다~" 처럼 붙어있는 형태

2. 고객사 담당자 비식별화
- 현재 조건: "홍길동 님"이나 "김부장님"을 → (고객사 담당자명 직함)님

> 문제점: 고객도 마이페어 직원에게 “홍길동 님”처럼 부를 수 있어 → 양방향 모두에 “님” 사용, 따라서 “님”만 보고 누가 누구인지 구분이 불가능

### 해결 전략
발화자의  sender_email 정보를 고려하자:
- 마이페어 이메일 도메인으로부터 발신자 역할(sender role) 추정 가능
- 예: sender_email이 @myfair.co.kr → 마이페어 직원
- 이 정보를 바탕으로 누가 말했는지 구분

In [18]:
## 1) 이름
def anonymize_message_v2(text, sender_email):
    if pd.isna(text):
        return text

    # 마이페어 상담원이 보낸 경우
    if "@myfair.co.kr" in sender_email:
        # 마이페어 + 이름 → 마이페어 (마이페어 담당자명)
        text = re.sub(r"마이페어\s[가-힣]{2,}(?=입니다|입니다\.)", "마이페어 (마이페어 담당자명)", text)

        # 고객사 담당자명 추정되는 "~~님"만 치환 (첫 번째 1회만)
        text = re.sub(r"[가-힣]{2,}(?:\s?[가-힣]{2,})?님", "(고객사 담당자명 직함)님", text, count=1)

    else:  # 고객사 담당자가 보낸 경우
        # 마이페어 상담원 이름이 있는 경우 → (마이페어 담당자명)으로 대체
        text = re.sub(r"마이페어\s[가-힣]{2,}님", "마이페어 (마이페어 담당자명)님", text)

        text = re.sub(r"[가-힣]{2,}(?:\s?[가-힣]{2,})?님", "(마이페어 담당자명)님", text, count=1)

    return text

In [19]:
ex = pd.DataFrame({
    "sender_email": [
        "agent01@myfair.co.kr",  # 마이페어 상담원
        "client@company.com",    # 고객사 담당자
        "agent02@myfair.co.kr",  # 마이페어 상담원
        "client@company.com",    # 고객사 담당자
    ],
    "message": [
        "김부장님, 안녕하세요. 마이페어 김지훈입니다.",
        "마이페어 김지훈님, 안녕하세요. 궁금한 게 있습니다.",
        "박과장님, 일정 확인 부탁드립니다. 마이페어 이민호입니다.",
        "이민호님, 감사합니다!",
    ]
})

# 적용
ex['message_anonymized'] = ex.apply(
    lambda row: anonymize_message_v2(row['message'], row['sender_email']),
    axis=1
)

In [20]:
# 출력 확인
print(ex[['message', 'message_anonymized']])

                            message  \
0         김부장님, 안녕하세요. 마이페어 김지훈입니다.   
1     마이페어 김지훈님, 안녕하세요. 궁금한 게 있습니다.   
2  박과장님, 일정 확인 부탁드립니다. 마이페어 이민호입니다.   
3                      이민호님, 감사합니다!   

                                  message_anonymized  
0        (고객사 담당자명 직함)님, 안녕하세요. 마이페어 (마이페어 담당자명)입니다.  
1              마이페어 (마이페어 담당자명)님, 안녕하세요. 궁금한 게 있습니다.  
2  (고객사 담당자명 직함)님, 일정 확인 부탁드립니다. 마이페어 (마이페어 담당자명)...  
3                               (마이페어 담당자명)님, 감사합니다!  


In [37]:
def anonymize_message_v3(text, sender_email):
    if pd.isna(text):
        return text

    # ===== 📌 공통 민감정보 비식별화 =====
    # 주민등록번호
    text = re.sub(r"\b\d{6}-\d{7}\b", "(주민등록번호)", text)

    # 카드번호 (공백 or 하이픈 구분)
    text = re.sub(r"(?:\d{4}[-\s]?){3}\d{4}", "(카드번호)", text)

    # 전화번호 (010-xxxx-xxxx, 02-xxx(x)-xxxx 등)
    text = re.sub(r"\b\d{2,3}-\d{3,4}-\d{4}\b", "(전화번호)", text)

    # 이메일 주소
    text = re.sub(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "(이메일)", text)

    # 주소 (서울시, 경기도 등 + 도로명 or 지번)
    text = re.sub(r"(서울|경기|부산|대구|인천|광주|대전|울산|세종|강원|충북|충남|전북|전남|경북|경남|제주)[^\s,]{0,20}(시|도)?\s?[^\s,]{1,20}(구|군)?[^\s,]{0,20}", "(주소)", text)

    # 영문 이름 (John Smith 등)
    text = re.sub(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b", "(영문이름)", text)

    # ===== 📌 보낸 사람별 역할 기반 이름 마스킹 =====
    if "@myfair.co.kr" in sender_email:
        # 마이페어 + 이름 → 마이페어 (담당자명)
        text = re.sub(r"마이페어\s[가-힣]{2,}(님)?", "마이페어 (마이페어 담당자명)\\1", text)

        # 고객사 이름 추정 "OOO님" (최대 1회)
        text = re.sub(r"[가-힣]{2,}(?:\s?[가-힣]{2,})?님", "(고객사 담당자명 직함)님", text, count=1)

    else:
        # 고객이 보낸 경우: 마이페어 이름 → 마스킹
        text = re.sub(r"마이페어\s[가-힣]{2,}님", "마이페어 (마이페어 담당자명)님", text)
        text = re.sub(r"[가-힣]{2,}(?:\s?[가-힣]{2,})?님", "(마이페어 담당자명)님", text, count=1)

    return text

In [38]:
ex2 = pd.DataFrame({
    "sender_email": [
        "agent1@myfair.co.kr",
        "client@example.com",
        "agent2@myfair.co.kr",
        "client2@company.com",
        "agent3@myfair.co.kr"
    ],
    "message": [
        "김부장님, 안녕하세요. 마이페어 김지훈입니다. 제 번호는 010-1234-5678입니다.",
        "마이페어 김지훈님, John Smith가 결제한 카드번호는 1234-5678-1234-5678입니다.",
        "박과장님, 일정 확인 부탁드립니다. 마이페어 이민호입니다. 이메일은 agent2@myfair.co.kr입니다.",
        "이민호님, 주소는 서울시 강남구 테헤란로123입니다. 주민번호는 900101-1234567입니다.",
        "고객님, 카드번호는 4321 8765 4321 8765이며, 연락처는 02-123-4567입니다."
    ]
})

In [41]:
ex2["message_anonymized"] = ex2.apply(
    lambda row: anonymize_message_v3(row["message"], row["sender_email"]),
    axis=1
)

In [43]:
# 출력 확인
ex2[['message', 'message_anonymized']]

,message,message_anonymized
0,"김부장님, 안녕하세요. 마이페어 김지훈입니다. 제 번호는 010-1234-5678입니다.","(고객사 담당자명 직함)님, 안녕하세요. 마이페어 (마이페어 담당자명). 제 번호는 010-1234-5678입니다."
1,"마이페어 김지훈님, John Smith가 결제한 카드번호는 1234-5678-1234-5678입니다.","마이페어 (마이페어 담당자명)님, John Smith가 결제한 카드번호는 (카드번호)입니다."
2,"박과장님, 일정 확인 부탁드립니다. 마이페어 이민호입니다. 이메일은 agent2@myfair.co.kr입니다.","(고객사 담당자명 직함)님, 일정 확인 부탁드립니다. 마이페어 (마이페어 담당자명). 이메일은 (이메일)입니다."
3,"이민호님, 주소는 서울시 강남구 테헤란로123입니다. 주민번호는 900101-1234567입니다.","(마이페어 담당자명)님, 주소는 (주소) 테헤란로123입니다. 주민번호는 900101-1234567입니다."
4,"고객님, 카드번호는 4321 8765 4321 8765이며, 연락처는 02-123-4567입니다.","(고객사 담당자명 직함)님, 카드번호는 (카드번호)이며, 연락처는 02-123-4567입니다."


***

# 영문 번역: Helsinki-NLP (MarianMT) + HuggingFace Transformers

In [1]:
!pip install transformers sentencepiece

     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     -------------------------------------  41.0/41.5 kB 653.6 kB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 496.9 kB/s eta 0:00:00
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
    --------------------------------------- 0.2/10.4 MB 4.2 MB/s eta 0:00:03
   -- ------------------------------------- 0.5/10.4 MB 6.7 MB/s eta 0:00:02
   -- ------------------------------------- 0.6/10.4 MB 5.8 MB/s eta 0:00:02
   -- ------------------------------------- 0.7/10.4 MB 4.4 MB/s eta 0:00:03
   -- ------------------------------------- 0.7/10.4 MB 4.4 MB/s eta 0:00:03
   -- ------------------------------------- 0.7/10.4 MB 4.4 MB/s eta 0:00:03
   --- ------------------------------------ 0.8/10.4 MB 2.7 MB/s eta 0:00:04
   ---- ----------------------------------- 1.1/10.4 MB 3.1 MB/s eta 0:00:03
   ----- 


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import MarianMTModel, MarianTokenizer

model_name = 'Helsinki-NLP/opus-mt-ko-en'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def translate(text):
    tokens = tokenizer.prepare_seq2seq_batch([text], return_tensors="pt")
    translation = model.generate(**tokens)
    return tokenizer.decode(translation[0], skip_special_tokens=True)

print(translate("안녕하세요. 반갑습니다."))

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

C:\Users\jwoo\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jwoo\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-ko-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

C:\Users\jwoo\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:4106: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)


Hello. Nice to meet you.


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

In [3]:
print(translate("담당자님 안녕하세요 보내주신 내용 확인했습니다 부재중이셔서 채팅드립니다"))

Hi, sir. I've checked your information for you, sir. You're absent, so we're talking to you.


In [6]:
print(translate("혹시 바우처 75만원 및 부가세 7만5천원 견적서가 있을까요? 부가세 결제는 보고용으로 필요할듯합니다 없으시면 채팅 내용 및 신청내용 캡쳐해서 사용해도 되니 편하게 말씀해주세요"))

If you don't need a report, please tell me you can use the information on the chats and requests.


In [7]:
print(translate("혹시 바우처 75만원 및 부가세 7만5천원 견적서가 있을까요? 부가세 결제는 보고용으로 필요할듯합니다"))

Do you have $75, U.S. and $75?
